# LLM Evaluation in Production
## One end-to-end customer-support story

Welcome to **MaviSepet**, a fictional online shop. Its assistant reads company policies, verifies orders, creates support tickets, and answers customers.

In this single notebook we will move one product from an untested baseline to a production-like A/B test. Nothing essential is hidden in another Python file.

By the end, you will know how to:

1. Define good behavior before changing a prompt.
2. Trace retrieval, generation, and tool actions separately.
3. Compare prompt A and B on the same evaluation dataset.
4. Route synthetic traffic with stable assignment.
5. Make a rollout, canary, or rollback decision.

### What does ‘production’ mean here?

We do not have real customer traffic. We simulate incoming requests and demonstrate production mechanics: variant assignment, metadata, traces, scores, guardrails, and rollback.

The results describe behavior on **this dataset**. They cannot prove an effect on real satisfaction, retention, or revenue.

## 0. Choose a safe workshop mode

- `demo` is deterministic, free, and reliable when conference Wi-Fi fails. It creates a local trace table.
- `live` calls an OpenRouter model and sends observations to Langfuse.

Both modes follow the same product path. Demo mode teaches mechanics; it is not evidence about a real LLM.

In [ ]:
import hashlib, json, os, random, time
from dataclasses import dataclass
from pathlib import Path
import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

MODE = os.getenv('WORKSHOP_MODE', 'demo').casefold()
MODEL_NAME = os.getenv('OPENROUTER_MODEL', 'openai/gpt-4.1-mini')
if MODE not in {'demo', 'live'}:
    raise ValueError('WORKSHOP_MODE must be demo or live')
print('Mode:', MODE)
print('Model:', MODEL_NAME if MODE == 'live' else 'deterministic teaching simulator')

**How to read this output:** `demo` means no model or Langfuse credentials are required. In `live`, model generations and nested observations appear in Langfuse.

## 1. Meet the product data

Our application uses three sources:

- **Policies:** the RAG knowledge base.
- **Orders:** private operational data accessed by a tool.
- **Evaluation cases:** customer inputs plus expected behavior.

Expected behavior belongs to the evaluator. The live application must never see it.

In [ ]:
locations = [Path('.'), Path('examples/04_customer_support'), Path('04_customer_support')]
ROOT = next((p for p in locations if (p / 'dataset.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Start Jupyter from the repository root or examples/04_customer_support.')
policies = json.loads((ROOT / 'policies.json').read_text())
orders = json.loads((ROOT / 'orders.json').read_text())
dataset = json.loads((ROOT / 'dataset.json').read_text())
print(f'{len(policies)} policies | {len(orders)} orders | {len(dataset)} evaluation cases')
pd.DataFrame(policies)[['id', 'title', 'text']]

In [ ]:
first_case = dataset[0]
print(json.dumps(first_case, indent=2, ensure_ascii=False))

### Read the evaluation contract

For this damaged-item request, a good system must retrieve the `damaged` policy, create a priority ticket, and avoid promising an instant refund.

The expected output describes **behavior**, not one perfect sentence. Many good phrasings can pass. This is more useful than exact string matching.

## 2. Define the experiment: one change at a time

A is the current baseline; B is the candidate. The model, data, retrieval, tools, and evaluators remain fixed. If we changed the model and prompt together, we could not identify the cause of a score change.

In [ ]:
@dataclass(frozen=True)
class Variant:
    name: str
    prompt: str
    simulated_latency_ms: int
    estimated_cost_usd: float

VARIANT_A = Variant('prod-a', 'Be a helpful customer-support assistant. Answer the customer.', 650, 0.0018)
VARIANT_B = Variant(
    'prod-b',
    'Use only supplied policy and verified order data. Never promise an instant refund or unverified delivery date. Acknowledge the problem, take the required action, and give one clear next step.',
    820, 0.0023,
)
pd.DataFrame([{'variant': v.name, 'prompt': v.prompt} for v in [VARIANT_A, VARIANT_B]])

**What changed?** B adds grounding, safety, and action instructions. It may improve quality, but it may also increase latency or cost. We therefore need several metrics, not one subjective answer review.

## 3. Build an observable application

One request has four boundaries:

1. Verify the private order.
2. Retrieve the relevant policy.
3. Generate text plus a structured action.
4. Create a ticket when required.

Returning intermediate outputs is essential: if we return only prose, we cannot tell whether retrieval, generation, or the action failed.

In [ ]:
TRACE_EVENTS, CREATED_TICKETS = [], []

def local_event(trace_id, step, **details):
    TRACE_EVENTS.append({'trace_id': trace_id, 'step': step, **details})

def identity_observe(*, name=None):
    return lambda function: function

observe_step, langfuse, callbacks = identity_observe, None, []
if MODE == 'live':
    required = ['OPENROUTER_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY']
    missing = [key for key in required if not os.getenv(key)]
    if missing:
        raise RuntimeError(f'Missing environment variables: {missing}')
    from langfuse import get_client, observe as observe_step
    from langfuse.langchain import CallbackHandler
    langfuse = get_client()
    callbacks = [CallbackHandler()]
    print('Langfuse connected:', langfuse.auth_check())

### 3.1 Verify the order — a tool call

The order lookup accesses structured operational data, not a RAG document. The assistant may see an order only when the supplied email matches.

In [ ]:
@observe_step(name='Verify Order')
def verify_order(order_id, email):
    order = next((row for row in orders if row['order_id'] == order_id), None)
    if order is None or order['email'].casefold() != email.casefold():
        return None
    return order

verify_order('ORD-1001', 'ada@example.com')

### 3.2 Retrieve policy context — the RAG step

A production RAG system may use embeddings and hybrid search. Transparent keyword retrieval is easier to inspect in a short workshop. The evaluation principle is identical: return document IDs and text so retrieval can be scored separately.

In [ ]:
@observe_step(name='Retrieve Policy')
def retrieve_policy(message, order_verified):
    text = message.casefold()
    if not order_verified:
        policy_id = 'privacy'
    elif any(w in text for w in ['broken', 'damaged']):
        policy_id = 'damaged'
    elif any(w in text for w in ['refund', 'bank', 'inspected']):
        policy_id = 'refunds'
    elif any(w in text for w in ['return', 'unused']):
        policy_id = 'returns'
    elif any(w in text for w in ['tracking', 'where is', 'delivery', 'arrive']):
        policy_id = 'delivery'
    else:
        policy_id = 'returns'
    return next(policy for policy in policies if policy['id'] == policy_id)

retrieve_policy(first_case['input']['message'], order_verified=True)

### 3.3 Generate text and a structured decision

A structured `create_ticket` field is cheaper and more reliable to evaluate than searching prose. The deterministic generator is a conference fallback: it reads product inputs, never expected answers. Live mode sends those same inputs to the configured model.

In [ ]:
def demo_generation(item, policy, order, variant):
    policy_id = policy['id']
    if variant.name == 'prod-b':
        options = {
            'damaged': ("I'm sorry your item arrived damaged. I created a priority support ticket; return shipping will be covered after review.", True),
            'delivery': ("I created a delivery investigation ticket because tracking is stale. I cannot guarantee an arrival date.", True),
            'returns': ("Unused products can be returned within 14 days after approval; a refund follows inspection.", False),
            'refunds': ("After inspection, the refund is sent in 5–7 business days; your bank may need more time.", False),
            'privacy': ("I could not verify this order. Please use a private support channel for identity verification.", False),
        }
    else:
        options = {
            'damaged': ("I can offer an instant refund for the damaged item.", False),
            'delivery': ("It will arrive tomorrow; please wait.", False),
            'returns': ("You can probably return it and receive a full refund.", False),
            'refunds': ("The money should appear immediately.", False),
            'privacy': ("The order details are available; I can show them here.", False),
        }
    answer, create_ticket = options[policy_id]
    return {'answer': answer, 'create_ticket': create_ticket}

def live_generation(item, policy, order, variant):
    from langchain_openrouter import ChatOpenRouter
    from pydantic import BaseModel, Field
    class Decision(BaseModel):
        answer: str = Field(description='Customer-facing answer')
        create_ticket: bool = Field(description='Whether a support ticket is required')
    model = ChatOpenRouter(model=MODEL_NAME, app_title='LLM Evaluation Workshop')
    prompt = f"{variant.prompt}\n\nPOLICY:\n{policy['text']}\n\nVERIFIED ORDER:\n{order or 'NOT VERIFIED'}\n\nCUSTOMER:\n{item['message']}"
    decision = model.with_structured_output(Decision).invoke(
        prompt, config={'callbacks': callbacks, 'tags': ['customer-support', variant.name], 'metadata': {'variant': variant.name}}
    )
    return {'answer': decision.answer, 'create_ticket': decision.create_ticket}

generate = live_generation if MODE == 'live' else demo_generation

### 3.4 Execute the action and return observable fields

The application returns the answer, retrieved policy, verification result, action, variant, latency, and cost. These fields let us attach specific evaluators and diagnose failures.

In [ ]:
@observe_step(name='Create Support Ticket')
def create_ticket(order_id, reason):
    ticket = {'ticket_id': f'T-{len(CREATED_TICKETS)+1:04}', 'order_id': order_id, 'reason': reason}
    CREATED_TICKETS.append(ticket)
    return ticket

@observe_step(name='Customer Support Pipeline')
def support_pipeline(item, variant):
    trace_id = f"{item['user_id']}:{variant.name}:{len(TRACE_EVENTS)}"
    started = time.perf_counter()
    order = verify_order(item['order_id'], item['email'])
    local_event(trace_id, 'verify_order', verified=order is not None)
    policy = retrieve_policy(item['message'], order_verified=order is not None)
    local_event(trace_id, 'retrieve_policy', policy_id=policy['id'])
    decision = generate(item, policy, order, variant)
    local_event(trace_id, 'generate', variant=variant.name, answer=decision['answer'])
    ticket = create_ticket(item['order_id'], policy['id']) if decision['create_ticket'] and order else None
    local_event(trace_id, 'create_ticket', created=ticket is not None)
    measured_ms = int((time.perf_counter() - started) * 1000)
    jitter = random.Random(f"{variant.name}:{item['user_id']}").randint(-35, 35)
    return {
        'trace_id': trace_id, 'answer': decision['answer'],
        'retrieved_policy_ids': [policy['id']], 'order_verified': order is not None,
        'ticket_created': ticket is not None, 'variant': variant.name,
        'latency_ms': measured_ms if MODE == 'live' else variant.simulated_latency_ms + jitter,
        'estimated_cost_usd': variant.estimated_cost_usd,
    }

## 4. Run one baseline request and debug its trace

Trying one request is useful for debugging, but not enough to claim quality. We start with one to learn how to inspect the path.

In [ ]:
TRACE_EVENTS.clear(); CREATED_TICKETS.clear()
baseline_output = support_pipeline(first_case['input'], VARIANT_A)
print(json.dumps(baseline_output, indent=2, ensure_ascii=False))
pd.DataFrame(TRACE_EVENTS)

**How to debug it:** retrieval found `damaged`, but generation promised an instant refund and the action step did not create a ticket. Retrieval is not the problem; changing embeddings would not fix this failure.

In live mode, open Langfuse and filter by the `customer-support` tag or variant. The number tells us something failed; the trace tells us where.

## 5. Turn expectations into deterministic metrics

We begin with cheap, explainable checks:

- `retrieval_recall`: did we retrieve every required policy?
- `answer_coverage`: are required next-step facts present?
- `policy_compliance`: did we avoid prohibited promises?
- `action_correctness`: did we create a ticket exactly when required?

We keep component scores even when calculating one summary score.

In [ ]:
def evaluate(output, expected):
    answer = output['answer'].casefold()
    needed, found = set(expected['policy_ids']), set(output['retrieved_policy_ids'])
    retrieval_recall = len(needed & found) / len(needed)
    answer_coverage = sum(term.casefold() in answer for term in expected['required_terms']) / len(expected['required_terms'])
    policy_compliance = float(not any(term.casefold() in answer for term in expected['forbidden_terms']))
    action_correctness = float(output['ticket_created'] == expected['create_ticket'])
    quality = (retrieval_recall + answer_coverage + policy_compliance + action_correctness) / 4
    return {'retrieval_recall': retrieval_recall, 'answer_coverage': answer_coverage, 'policy_compliance': policy_compliance, 'action_correctness': action_correctness, 'quality': quality}

pd.Series(evaluate(baseline_output, first_case['expected_output']), name='baseline')

`quality` is convenient for comparison; its components explain what to fix. In a high-risk product, policy compliance should be a hard guardrail rather than merely one quarter of an average.

## 6. Run a controlled offline A/B experiment

Every case goes through both variants. This is a **paired offline experiment**, not randomized production traffic. Both variants receive identical inputs, so regressions are easy to locate.

In [ ]:
offline_rows = []
for case in dataset:
    for variant in [VARIANT_A, VARIANT_B]:
        output = support_pipeline(case['input'], variant)
        offline_rows.append({'user_id': case['input']['user_id'], 'message': case['input']['message'], **output, **evaluate(output, case['expected_output'])})
offline = pd.DataFrame(offline_rows)
metrics = ['retrieval_recall', 'answer_coverage', 'policy_compliance', 'action_correctness', 'quality']
offline.groupby('variant')[metrics + ['latency_ms', 'estimated_cost_usd']].mean().round(4)

**Read the table in both directions:** higher quality metrics are better; lower latency and cost are better. B must pass policy and action guardrails without an unacceptable operational trade-off.

In [ ]:
# Averages hide regressions. Inspect failures before trusting a winner.
offline.loc[offline['quality'] < 1, ['variant','message','answer','retrieval_recall','policy_compliance','action_correctness','quality']].sort_values(['variant','quality'])

### What did this prove?

It compared A and B on known cases with expected behavior—enough to reject obvious regressions before deployment. It did not expose real users or measure their behavior.

In Langfuse these cases can also be uploaded as a dataset and run with `dataset.run_experiment(...)`. We keep the loop visible here so beginners can see exactly what an experiment runner does.

## 7. Simulate production A/B routing

After offline checks, incoming users can be split between A and B. A user must not switch variants on every request. We hash a stable user ID into bucket 0 or 1. A feature-flag service would usually own this in production.

In [ ]:
def stable_variant(user_id):
    bucket = int(hashlib.sha256(user_id.encode()).hexdigest()[:8], 16) % 2
    return [VARIANT_A, VARIANT_B][bucket]

assignments = pd.DataFrame([{'user_id': c['input']['user_id'], 'variant': stable_variant(c['input']['user_id']).name} for c in dataset])
assignments

Run the cell again: every user stays in the same group. With only ten users, groups may not be exactly equal. Randomization balances groups in expectation, not necessarily in a tiny sample.

In [ ]:
traffic_rows = []
for cycle in range(5):
    for case in dataset:
        variant = stable_variant(case['input']['user_id'])
        output = support_pipeline(case['input'], variant)
        traffic_rows.append({'cycle': cycle, 'user_id': case['input']['user_id'], **output, **evaluate(output, case['expected_output'])})
traffic = pd.DataFrame(traffic_rows)
online_summary = traffic.groupby('variant').agg(
    requests=('user_id','size'), users=('user_id','nunique'), quality=('quality','mean'),
    policy_compliance=('policy_compliance','mean'), action_correctness=('action_correctness','mean'),
    latency_ms=('latency_ms','mean'), estimated_cost_usd=('estimated_cost_usd','mean'),
).round(4)
online_summary

### Offline and online evaluation complement each other

Offline evaluation has reference answers and protects deployment. Online evaluation sees real input diversity but often lacks references. Production metrics may include latency, cost, feedback, escalation, and business outcomes.

Our simulated stream still has expected outputs because it is a workshop. In a real stream, reference-based checks would be sampled or replaced with feedback, guardrails, business outcomes, and calibrated judges.

## 8. Decide before moving the goalposts

Agree on thresholds before reading results. Example rules:

1. Roll back when policy compliance is below 98%.
2. Canary when quality is below 85%.
3. Canary when mean latency exceeds 1,200 ms.
4. Roll out only when every guardrail passes.

These thresholds are illustrative, not universal.

In [ ]:
def rollout_decision(candidate):
    if candidate['policy_compliance'] < 0.98:
        return 'ROLLBACK — policy compliance is below 98%.'
    if candidate['quality'] < 0.85:
        return 'CANARY — quality is below the full-rollout threshold.'
    if candidate['latency_ms'] > 1200:
        return 'CANARY — latency needs investigation.'
    return 'ROLLOUT — all workshop guardrails pass.'

candidate = offline.query("variant == 'prod-b'").agg({'policy_compliance':'mean','quality':'mean','latency_ms':'mean'}).to_dict()
print(candidate)
print(rollout_decision(candidate))
if langfuse is not None:
    langfuse.flush()

## 9. Workshop versus production

| Workshop component | Production counterpart |
| --- | --- |
| JSON orders | Authenticated order service |
| Keyword retrieval | Versioned vector or hybrid retrieval |
| Python hash | Feature-flag and experimentation platform |
| Ten labelled cases | Versioned regression set plus sampled real traces |
| Local score table | Langfuse scores, dashboards, and alerts |
| Fixed cost estimate | Observed token usage and provider cost |
| Manual decision | Canary automation and rollback runbook |

Real traffic also requires privacy review, data-retention rules, incident ownership, and a plan for harmful outputs.

# Takeaways

1. Define good behavior before changing the prompt.
2. Keep retrieval, generation, and actions separately observable.
3. Use paired offline experiments before exposing users.
4. Change one variable at a time.
5. Use stable assignment for online A/B traffic.
6. Read failed traces, not only averages.
7. Treat safety as a guardrail.
8. Synthetic traffic proves mechanics—not customer impact.

> **The number tells us that something changed. The trace helps us find where.**